In [ ]:
#Imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq, ifft
from scipy.signal import windows
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import io, os
import base64
# Runs on plain Jupyter/Binder/Voila as well as Colab: uses only
# ipywidgets.FileUpload for loading and a data-URI <a download> link
# for exporting, instead of google.colab-specific APIs.

In [8]:
# Configuration settings for spectrum processing
class ProcessConfig:
    def __init__(self, normalize_range=None, apply_log = False):
        self.normalize_range = normalize_range
        self.apply_log = apply_log

In [9]:
# Container for FFT output
class FFTData:
    def __init__(self, canal, values):
        self.canal = canal
        self.values = values

In [10]:
class SpectrumData:
    def __init__(self, x, y, config: ProcessConfig):
        self.x = x
        self.y = y
        self.config = config

        self.fft = None
        self.current_fft = None
        self.log_fft = None
        self.ifft = None

        self.window_used = None
        self.deconv_filter = None

    # ---- File loading ----

    @staticmethod
    def parse_jdx(text: str):
        """Parse a JCAMP-DX file (already decoded to text) and return (x, y, metadata)."""
        firstx = lastx = deltax = None
        xfactor = yfactor = 1.0
        ys = []
        in_data = False
        for line in text.splitlines():
            s = line.strip()
            if s.startswith("##"):
                in_data = False
                key, _, val = s[2:].partition("=")
                key = key.strip().upper()
                val = val.strip()
                if key == "FIRSTX": firstx = float(val)
                elif key == "LASTX": lastx = float(val)
                elif key == "DELTAX": deltax = float(val)
                elif key == "XFACTOR": xfactor = float(val)
                elif key == "YFACTOR": yfactor = float(val)
                elif key in ("XYDATA", "XYPOINTS", "PEAK TABLE", "PEAKTABLE"):
                    in_data = True
                continue
            if in_data and s:
                nums = s.replace(",", " ").split()
                try:
                    vals = [float(n) for n in nums]
                except ValueError:
                    continue
                ys.extend(vals[1:])
        if ys and firstx is not None and (lastx is not None or deltax is not None):
            y = np.array(ys) * yfactor
            n = len(y)
            x = (np.linspace(firstx, lastx, n) if lastx is not None
                 else firstx + deltax * np.arange(n)) * xfactor
            return x, y, {}
        raise ValueError("JCAMP parse failed")

    @staticmethod
    def load_spectrum(raw_bytes: bytes, filename: str):
        """Load a spectrum from raw file bytes.

        Supports .jdx/.dx (JCAMP-DX) and generic two-column text files
        (whitespace, comma, semicolon, or tab separated; '#', ';', '%%', '>' lines
        are treated as comments).
        """
        text = raw_bytes.decode("utf-8", errors="ignore") if isinstance(raw_bytes, (bytes, bytearray)) else raw_bytes

        if filename.lower().endswith((".jdx", ".dx")):
            try:
                x, y, meta = SpectrumData.parse_jdx(text)
                meta = {**meta, "title": filename}
                return x, y, meta
            except Exception:
                pass  # fall through and try the generic text parser

        xs, ys = [], []
        for line in text.splitlines():
            s = line.strip()
            if not s or s[0] in "#;%>":
                continue
            parts = s.replace(",", " ").replace(";", " ").replace("\t", " ").split()
            try:
                vals = [float(p) for p in parts[:2]]
            except ValueError:
                continue
            if len(vals) >= 2:
                xs.append(vals[0])
                ys.append(vals[1])
        if len(xs) < 2:
            raise ValueError("Could not parse two numeric columns.")
        metadata = {"title": filename, "xunits": "", "yunits": "", "data_type": "TEXT"}
        return np.array(xs, float), np.array(ys, float), metadata

    def preprocess(self):
        self.x_raw = self.x.copy()
        self.y_raw = self.y.copy()

        x_uniform = np.linspace(self.x.min(), self.x.max(), len(self.x))
        y_uniform = np.interp(x_uniform, self.x, self.y)

        y_uniform -= np.mean(y_uniform)

        self.x = x_uniform
        self.y = y_uniform

    def compute_fft(self):
        n = len(self.y)
        dx = np.mean(np.diff(self.x))
        canal = fftfreq(n, d=dx)          # fix 1: was fftcanal
        values = fft(self.y)

        self.fft = FFTData(canal, values)
        self.current_fft = FFTData(canal.copy(), values.copy())

    def compute_log_fft(self):
        if self.fft is None:
            raise ValueError("Run compute_fft first")

        amp = np.abs(self.fft.values)
        amp_ref = np.max(amp)

        if amp_ref == 0:
            raise ValueError("FFT is all zeros")

        ratio = np.maximum(amp / amp_ref, 1e-12)
        self.log_fft = (10 * np.log10(ratio))

    def apodize(self, window_type='Gaussian', L=None, sigma=None):
        if self.current_fft is None:
            raise ValueError("Run compute_fft first")

        canal = self.current_fft.canal
        window = self.build_apod_window(canal, window_type, L, sigma)

        self.window_used = window
        self.current_fft = FFTData(canal, self.current_fft.values * window)

    def deconvolve_with_apodization(
        self,
        delta=0.01,
        rho=1.5,
        window_type='Gaussian',
        sigma=None,
        L=None,
    ):
        if self.fft is None:
            raise ValueError("Run compute_fft first")

        canal = self.fft.canal
        canal_abs = np.abs(canal)

        H = np.exp(-delta * (canal_abs ** rho))
        eps = 1e-12
        deconv_filter = 1.0 / (H + eps)

        apod_window = self.build_apod_window(canal, window_type, L, sigma)

        combined_filter = deconv_filter * apod_window

        self.deconv_filter = deconv_filter
        self.window_used = apod_window
        self.current_fft = FFTData(canal, self.fft.values * combined_filter)  # fix 2: was freq

    def build_apod_window(self, canal, window_type, L, sigma):
        canal_abs = np.abs(canal)
        canal_max = np.max(canal_abs)

        if L is None:
            if sigma is None:
                sigma = 0.10
            L = sigma * canal_max

        if window_type == 'Boxcar':
            window = self.apod_boxcar(canal_abs, L)
        elif window_type == 'Gaussian':
            window = self.apod_gaussian(canal_abs, L)
        elif window_type == 'Sigmoid':
            window = self.apod_sigmoid(canal_abs, L)
        elif window_type == 'Bessel':
            window = self.apod_bessel(canal_abs, L)
        elif window_type == 'Sync2':
            window = self.apod_sync2(canal_abs, L)
        else:
            raise ValueError(
                f"Unknown window type '{window_type}'. "
                "Choose from: Boxcar, Gaussian, Sigmoid, Bessel, Sync2"
            )

        window[0] = 1.0
        return window

    @staticmethod
    def apod_boxcar(canal_abs, L):
        """D(u, L) = 1 if u < L, else 0"""
        return np.where(canal_abs < L, 1.0, 0.0)

    @staticmethod
    def apod_gaussian(canal_abs, L):
        """D(u, L) = exp(-u² / L²)"""
        return np.exp(-(canal_abs ** 2) / (L ** 2))

    @staticmethod
    def apod_sigmoid(canal_abs, L, b=10.0):
        """D(u, L) = 1 / (1 + exp(b * (u - L)))"""
        return 1.0 / (1.0 + np.exp(b * (canal_abs - L)))

    @staticmethod
    def apod_bessel(canal_abs, L):           # fix 3: parameter was freq_abs, used canal_abs
        """D(u, L) = (1 - u²/L²)²  if u < L, else 0"""
        return np.where(canal_abs < L, (1.0 - (canal_abs / L) ** 2) ** 2, 0.0)

    @staticmethod
    def apod_sync2(canal_abs, L):            # fix 4: parameter was inconsistently named
        """D(u, L) = sinc²(a*u) / (a*u)²  if u < L, else 0  where a = π/L"""
        a = np.pi / L
        au = a * canal_abs
        with np.errstate(invalid='ignore', divide='ignore'):
            window = np.where(au == 0, 1.0, (np.sin(au) / au) ** 2)
        window = np.where(canal_abs < L, window, 0.0)
        return window

    def compute_ifft(self):
        if self.current_fft is None:
            raise ValueError("No FFT available")

        self.ifft = np.real(ifft(self.current_fft.values))

In [11]:
# # Spectrum File Loader .txt / .csv / .dat / .asc / .jdx / .dx

# def parse_jdx(text: str):
#     """Parse a JCAMP-DX file (already decoded to text) and return (x, y, metadata)."""
#     firstx = lastx = deltax = None
#     xfactor = yfactor = 1.0
#     ys = []
#     in_data = False
#     for line in text.splitlines():
#         s = line.strip()
#         if s.startswith("##"):
#             in_data = False
#             key, _, val = s[2:].partition("=")
#             key = key.strip().upper()
#             val = val.strip()
#             if key == "FIRSTX": firstx = float(val)
#             elif key == "LASTX": lastx = float(val)
#             elif key == "DELTAX": deltax = float(val)
#             elif key == "XFACTOR": xfactor = float(val)
#             elif key == "YFACTOR": yfactor = float(val)
#             elif key in ("XYDATA", "XYPOINTS", "PEAK TABLE", "PEAKTABLE"):
#                 in_data = True
#             continue
#         if in_data and s:
#             nums = s.replace(",", " ").split()
#             try:
#                 vals = [float(n) for n in nums]
#             except ValueError:
#                 continue
#             ys.extend(vals[1:])
#     if ys and firstx is not None and (lastx is not None or deltax is not None):
#         y = np.array(ys) * yfactor
#         n = len(y)
#         x = (np.linspace(firstx, lastx, n) if lastx is not None
#              else firstx + deltax * np.arange(n)) * xfactor
#         return x, y, {}
#     raise ValueError("JCAMP parse failed")


# def load_spectrum(raw_bytes: bytes, filename: str):
#     """Load a spectrum from raw file bytes.

#     Supports .jdx/.dx (JCAMP-DX) and generic two-column text files
#     (whitespace, comma, semicolon, or tab separated; '#', ';', '%%', '>' lines
#     are treated as comments).
#     """
#     text = raw_bytes.decode("utf-8", errors="ignore") if isinstance(raw_bytes, (bytes, bytearray)) else raw_bytes

#     if filename.lower().endswith((".jdx", ".dx")):
#         try:
#             x, y, meta = parse_jdx(text)
#             meta = {**meta, "title": filename}
#             return x, y, meta
#         except Exception:
#             pass  # fall through and try the generic text parser

#     xs, ys = [], []
#     for line in text.splitlines():
#         s = line.strip()
#         if not s or s[0] in "#;%>":
#             continue
#         parts = s.replace(",", " ").replace(";", " ").replace("\t", " ").split()
#         try:
#             vals = [float(p) for p in parts[:2]]
#         except ValueError:
#             continue
#         if len(vals) >= 2:
#             xs.append(vals[0])
#             ys.append(vals[1])
#     if len(xs) < 2:
#         raise ValueError("Could not parse two numeric columns.")
#     metadata = {"title": filename, "xunits": "", "yunits": "", "data_type": "TEXT"}
#     return np.array(xs, float), np.array(ys, float), metadata

#**GUI**

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="Consider using IPython.display.IFrame")

# NOTE: ProcessConfig, FFTData, SpectrumData, parse_jdx and load_spectrum are
# defined in cells 1-4 above. This cell only builds the GUI on top of them,
# so edits/tests to the processing classes belong in those cells, not here.

import numpy as np
import io
import base64
import html as html_lib
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import plotly.graph_objects as go

PLOT_CONFIG = {"scrollZoom": True, "displaylogo": False,
               "modeBarButtonsToRemove": ["lasso2d", "select2d"]}


class SpectrumGUI:
    WINDOWS = ["Gaussian", "Boxcar", "Sigmoid", "Bessel", "Sync2"]

    def __init__(self):
        self.spec = None
        self.mode = "apod"
        self._applied = False
        self._busy = False
        self._panel_open = True

        # loader (plain ipywidgets.FileUpload - works on Colab, Binder,
        # Voila, and plain Jupyter alike, no platform-specific bridge needed)
        self.out_loader = widgets.Output()
        self.uploader = None
        self.btn_clear = widgets.Button(description="Clear File", button_style="danger",
                                        icon="trash", disabled=True)
        self.lbl_file = widgets.Label("No file loaded")

        # base views
        BW = widgets.Layout(width="340px")
        self.tg_orig = widgets.ToggleButton(description="Original Spectra", layout=BW)
        self.tg_fft = widgets.ToggleButton(description="FFT", layout=BW)
        self.tg_lfft = widgets.ToggleButton(description="LOG FFT", layout=BW)

        # toolbar
        self.btn_apply = widgets.Button(description="Apply", button_style="success",
                                        layout=widgets.Layout(width="90px"))
        self.tg_brpoint = widgets.ToggleButton(description="Break Point",
                                               layout=widgets.Layout(width="110px"))
        self.dd_win = widgets.Dropdown(options=self.WINDOWS, value="Gaussian",
                                       layout=widgets.Layout(width="150px"))
        self.btn_close = widgets.Button(description="Close", layout=widgets.Layout(width="80px"))

        # mode
        self.rb_mode = widgets.RadioButtons(
            options=[("Apodization", "apod"), ("Apodization and Deconvolution", "deconv")],
            value="apod", description="Mode:", style={"description_width": "60px"})

        # sliders
        self.sl_sigma = widgets.FloatSlider(value=0.64, min=0.01, max=1.0, step=0.01,
                                            description="sigma:", readout_format=".2f",
                                            continuous_update=False,
                                            layout=widgets.Layout(width="620px"))
        self.sl_delta = widgets.FloatSlider(value=65.0, min=0.0, max=200.0, step=1.0,
                                            description="delta", readout_format=".0f",
                                            orientation="vertical", continuous_update=False,
                                            layout=widgets.Layout(height="150px"))
        self.sl_rho = widgets.FloatSlider(value=1.65, min=1.0, max=2.0, step=0.05,
                                          description="rho", readout_format=".2f",
                                          orientation="vertical", continuous_update=False,
                                          layout=widgets.Layout(height="150px"))
        self.lbl_apod = widgets.Label("Apod.Point = 64%")

        # result views
        VW = widgets.Layout(width="340px")
        self.tg_mfft = widgets.ToggleButton(description="Show Modified FFT Signal", disabled=True, layout=VW)
        self.tg_cmpfft = widgets.ToggleButton(description="Show Original vs Modified FFT Signal", disabled=True, layout=VW)
        self.tg_difffft = widgets.ToggleButton(description="Show Difference Between FFT Signals", disabled=True, layout=VW)
        self.tg_msig = widgets.ToggleButton(description="Show Modified Signal", disabled=True, layout=VW)
        self.tg_cmpsig = widgets.ToggleButton(description="Show Original and Modified Signal", disabled=True, layout=VW)
        self.tg_diffsig = widgets.ToggleButton(description="Show Difference Between Signals", disabled=True, layout=VW)

        self.btn_export = widgets.Button(description="Export Signal", button_style="warning",
                                         icon="download", disabled=True)

        # outputs
        self.out_norm = widgets.Output(layout=widgets.Layout(width="780px", min_height="300px"))
        self.out_status = widgets.Output()
        self.out_plot = widgets.Output()

        self._base_toggles = [self.tg_orig, self.tg_fft, self.tg_lfft]
        self._proc_toggles = [self.tg_mfft, self.tg_cmpfft, self.tg_difffft,
                              self.tg_msig, self.tg_cmpsig, self.tg_diffsig]
        self._all_toggles = self._base_toggles + self._proc_toggles

        # events
        self.btn_clear.on_click(self._clear)
        self.btn_apply.on_click(self._apply)
        self.btn_close.on_click(self._toggle_panel)
        self.btn_export.on_click(self._export)
        self.rb_mode.observe(lambda c: self._on_mode_change(c["new"]), names="value")
        self.dd_win.observe(lambda c: self._on_window_change(), names="value")
        for s in (self.sl_sigma, self.sl_delta, self.sl_rho):
            s.observe(lambda c: self._on_param_change(), names="value")
        self.tg_brpoint.observe(lambda c: self._draw_norm(), names="value")
        for t in self._all_toggles:
            t.observe(lambda c: self._draw(), names="value")

        self._render_loader(active=True)
        self._on_mode_change("apod")

    def _msg(self, text, color="green"):
        with self.out_status:
            clear_output(wait=True)
            display(HTML(f"<span style='color:{color};font-weight:bold'>{text}</span>"))

    def _reset_current(self):  # fresh copy
        self.spec.current_fft = FFTData(self.spec.fft.canal.copy(), self.spec.fft.values.copy())

    def _set_toggles_off(self):
        self._busy = True
        for t in self._all_toggles:
            t.value = False
        self._busy = False

    def _fig(self, title, xlab, ylab, height=340, width=740, hovermode="closest"):
        fig = go.Figure()
        fig.update_layout(template="plotly_white", height=height, width=width,
                          margin=dict(l=55, r=15, t=45, b=40), title=title,
                          xaxis_title=xlab, yaxis_title=ylab, dragmode="zoom",
                          hovermode=hovermode,
                          legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0))
        return fig

    def _show_iframe(self, out, figs, per=390):  # zoomable iframe
        if not figs:
            with out:
                clear_output(wait=False)
            return
        divs = [f.to_html(full_html=False, include_plotlyjs=(i == 0), config=PLOT_CONFIG)
                for i, f in enumerate(figs)]
        page = ("<!doctype html><html><head><meta charset='utf-8'></head>"
                "<body style='margin:0'>" + "".join(divs) + "</body></html>")
        srcdoc = html_lib.escape(page, quote=True)
        h = per * len(figs) + 20
        with out:
            clear_output(wait=True)
            display(HTML(f'<iframe srcdoc="{srcdoc}" width="800" height="{h}" '
                         f'style="border:none;"></iframe>'))

    def _fft_xhi(self):  # crop tail
        f = self.spec.fft
        m = f.canal > 0
        c, a = f.canal[m], np.abs(f.values[m])
        if a.size == 0 or a.max() <= 0:
            return float(f.canal[m].max()) if a.size else 1.0
        idx = np.where(a > a.max() * 1e-3)[0]
        return float(c[idx[-1]]) * 1.1 if idx.size else float(c.max())

    def _make_uploader(self):
        up = widgets.FileUpload(
            accept=".txt,.csv,.dat,.asc,.jdx,.dx",
            multiple=False,
            description="Load file",
            button_style="primary",
            layout=widgets.Layout(width="160px"),
        )
        up.observe(self._on_upload_change, names="value")
        return up

    def _render_loader(self, active):
        with self.out_loader:
            clear_output(wait=True)
            if active:
                # A brand new FileUpload widget every time (rather than
                # reusing one and clearing .value) sidesteps version
                # differences in how FileUpload.value is shaped across
                # ipywidgets releases, and guarantees selecting the same
                # filename again after Clear still fires a change event.
                self.uploader = self._make_uploader()
                display(self.uploader)
            else:
                self.uploader = None
                display(HTML("""
                <span style="display:inline-block;background:#9e9e9e;color:#fff;padding:6px 18px;
                             border-radius:4px;font-weight:bold;font-family:sans-serif;cursor:not-allowed;">
                  File loaded (use Clear File)
                </span>"""))

    @staticmethod
    def _unpack_upload(new_value):
        """Normalize ipywidgets.FileUpload.value across widget versions.

        ipywidgets >= 8: a tuple/list of dicts, each with 'name'/'content'.
        ipywidgets < 8:  a dict keyed by filename, each value a dict with
        'metadata'/'content'.
        """
        if isinstance(new_value, dict):
            name, info = next(iter(new_value.items()))
            content = info["content"]
        else:
            item = new_value[0]
            name = item["name"]
            content = item["content"]
        return name, bytes(content)

    def _on_upload_change(self, change):
        new_value = change.get("new")
        if not new_value or self.spec is not None or self._busy:  # one request
            return
        self._busy = True
        try:
            name, raw = self._unpack_upload(new_value)
            x, y, meta = SpectrumData.load_spectrum(raw, name)
            self.spec = SpectrumData(np.asarray(x, float), np.asarray(y, float), ProcessConfig())
            self.spec.preprocess()
            self.spec.compute_fft()
            self.spec.compute_log_fft()
            self._applied = False
            self.lbl_file.value = f"Loaded: {name}  ({len(x)} points)"
            self._render_loader(active=False)  # lock
            self.btn_clear.disabled = False
            for t in self._proc_toggles:
                t.disabled = True
            self.btn_export.disabled = True
            self.tg_orig.value = True
            self._msg(f"Loaded: {name}", "green")
            self._draw()
            self._draw_norm()
        except Exception as e:
            self.spec = None
            self._msg(f"Load failed: {e}", "red")
        finally:
            self._busy = False

    def _clear(self, _):  # reset
        self.spec = None
        self._applied = False
        self.lbl_file.value = "No file loaded"
        self._set_toggles_off()
        for t in self._proc_toggles:
            t.disabled = True
        self.btn_export.disabled = True
        self.btn_clear.disabled = True
        self._render_loader(active=True)
        with self.out_plot:
            clear_output(wait=False)
        with self.out_norm:
            clear_output(wait=False)
        self._msg("Cleared. You can load a new file.", "purple")

    def _on_mode_change(self, mode):
        self.mode = mode
        show = (mode == "deconv")
        self.sl_delta.layout.display = "flex" if show else "none"
        self.sl_rho.layout.display = "flex" if show else "none"
        self._draw_norm()

    def _on_window_change(self):
        if self.spec is None:
            return
        self.tg_lfft.value = True  # auto log fft
        self._draw()
        self._draw_norm()

    def _on_param_change(self):
        self.lbl_apod.value = f"Apod.Point = {int(round(self.sl_sigma.value*100))}%"
        self._draw_norm()

    def _toggle_panel(self, _):
        self._panel_open = not self._panel_open
        self.btn_close.description = "Close" if self._panel_open else "Open"
        if self._panel_open:
            self._draw_norm()
        else:
            with self.out_norm:
                clear_output(wait=False)

    def _apply(self, _):
        if self.spec is None:
            self._msg("Load a file first.", "red")
            return
        try:
            win, sigma = self.dd_win.value, self.sl_sigma.value
            if self.mode == "apod":
                self._reset_current()
                self.spec.deconv_filter = None
                self.spec.apodize(window_type=win, sigma=sigma)
            else:
                self.spec.deconvolve_with_apodization(
                    delta=self.sl_delta.value, rho=self.sl_rho.value, window_type=win, sigma=sigma)
            self.spec.compute_ifft()
            self._applied = True
            for t in self._proc_toggles:
                t.disabled = False
            self.btn_export.disabled = False
            label = "Apodization" if self.mode == "apod" else "Apodization and Deconvolution"
            self._msg(f"{label} applied ('{win}').", "green")
            self.tg_mfft.value = True
            self._draw()
            self._draw_norm()
        except Exception as e:
            self._msg(f"Apply failed: {e}", "red")

    def _export(self, _):  # save txt
        if self.spec is None or self.spec.ifft is None:
            self._msg("Apply a window first.", "red")
            return
        try:
            # No google.colab.files.download here (Binder/Voila have no
            # such bridge). Instead, build the file in memory and offer it
            # as a browser download link via a base64 data URI - this
            # works the same in Colab, Binder, Voila, and plain Jupyter.
            out = np.column_stack((self.spec.x, self.spec.ifft))
            buf = io.StringIO()
            np.savetxt(buf, out, fmt="%.6f", header="Wavenumber Intensity")
            payload = buf.getvalue().encode("utf-8")
            b64 = base64.b64encode(payload).decode("ascii")
            href = f"data:text/plain;base64,{b64}"
            with self.out_status:
                clear_output(wait=True)
                display(HTML(
                    f"<a download='processed_signal.txt' href='{href}' "
                    "style='display:inline-block;background:#2e7d32;color:#fff;"
                    "padding:6px 14px;border-radius:4px;font-weight:bold;"
                    "text-decoration:none;font-family:sans-serif;'>"
                    "Download processed_signal.txt</a>"
                ))
        except Exception as e:
            self._msg(f"Export failed: {e}", "red")

    def _draw_norm(self):  # overlay curves
        if self.spec is None or not self._panel_open:
            return
        s = self.spec
        if s.fft is None: s.compute_fft()
        if s.log_fft is None: s.compute_log_fft()
        canal = s.fft.canal; ca = np.abs(canal); m = canal > 0
        win = s.build_apod_window(canal, self.dd_win.value, None, self.sl_sigma.value)

        fig = self._fig("Normalized Domain", "Channel", "db", height=380, width=740)
        fig.add_scatter(x=canal[m], y=s.log_fft[m], mode="lines", name="log FFT")
        wdb = 10 * np.log10(np.maximum(win[m], 1e-12))
        fig.add_scatter(x=canal[m], y=wdb, mode="lines", name="apodization curve", line=dict(dash="dash"))
        if self.mode == "deconv":
            # Preview curve: the decaying response function H itself (not its
            # inverse 1/H). The applied filter in deconvolve_with_apodization
            # still uses 1/H as the actual inverse filter - only this overlay
            # line was showing the wrong (increasing) shape.
            H = np.exp(-self.sl_delta.value * (ca ** self.sl_rho.value))
            hdb = 10 * np.log10(np.maximum(H[m], 1e-12))
            fig.add_scatter(x=canal[m], y=hdb, mode="lines",
                            name="deconvolution curve", line=dict(dash="dashdot"))
        if self.tg_brpoint.value:
            fig.add_vline(x=self.sl_sigma.value * np.max(ca), line_dash="dot", line_color="firebrick")
        fig.update_xaxes(range=[0, self._fft_xhi()])
        lo = float(np.min(s.log_fft[m])); hi = float(np.max(s.log_fft[m]))
        pad = (hi - lo) * 0.15 + 1e-6
        fig.update_yaxes(range=[lo - pad, hi + pad])
        self._show_iframe(self.out_norm, [fig], per=410)

    def _fig_original(self):
        f = self._fig("Original Signal", "Wavenumber (cm-1)", "Intensity (a.u.)")
        f.add_scatter(x=self.spec.x, y=self.spec.y, mode="lines")
        f.update_xaxes(range=[float(self.spec.x.min()), float(self.spec.x.max())])
        return f

    def _fig_fft(self):
        m = self.spec.fft.canal > 0
        f = self._fig("FFT Spectrum", "OPD / Canal (cm)", "Amplitude")
        f.add_scatter(x=self.spec.fft.canal[m], y=np.abs(self.spec.fft.values[m]), mode="lines")
        f.update_xaxes(range=[0, self._fft_xhi()])
        return f

    def _fig_lfft(self):
        m = self.spec.fft.canal > 0
        f = self._fig("Log FFT", "OPD / Canal (cm)", "Amplitude (dB)")
        f.add_scatter(x=self.spec.fft.canal[m], y=self.spec.log_fft[m], mode="lines")
        f.update_xaxes(range=[0, self._fft_xhi()])
        return f

    def _fig_mod_fft(self):
        m = self.spec.current_fft.canal > 0
        f = self._fig("Modified FFT Signal", "OPD / Canal (cm)", "Amplitude")
        f.add_scatter(x=self.spec.current_fft.canal[m], y=np.abs(self.spec.current_fft.values[m]), mode="lines")
        f.update_xaxes(range=[0, self._fft_xhi()])
        return f

    def _fig_cmp_fft(self):  # faded original
        s = self.spec; m = s.fft.canal > 0
        f = self._fig("Original vs Modified FFT", "OPD / Canal (cm)", "Amplitude")
        f.add_scatter(x=s.fft.canal[m], y=np.abs(s.fft.values[m]), mode="lines",
                      name="Original FFT", line=dict(color="gray", width=3), opacity=0.4)
        f.add_scatter(x=s.fft.canal[m], y=np.abs(s.current_fft.values[m]), mode="lines",
                      name="Modified FFT", line=dict(color="crimson", width=1.6))
        f.update_xaxes(range=[0, self._fft_xhi()])
        return f

    def _fig_diff_fft(self):
        s = self.spec; m = s.fft.canal > 0
        diff = np.abs(s.current_fft.values[m]) - np.abs(s.fft.values[m])
        f = self._fig("Difference Between FFT Signals", "OPD / Canal (cm)", "Amplitude Difference")
        f.add_scatter(x=s.fft.canal[m], y=diff, mode="lines")
        f.add_hline(y=0, line_dash="dash", line_color="black")
        f.update_xaxes(range=[0, self._fft_xhi()])
        return f

    def _fig_mod_sig(self):
        f = self._fig("Modified Signal (Inverse FFT)", "Wavenumber (cm-1)", "Intensity (a.u.)")
        f.add_scatter(x=self.spec.x, y=self.spec.ifft, mode="lines")
        f.update_xaxes(range=[float(self.spec.x.min()), float(self.spec.x.max())])
        return f

    def _fig_cmp_sig(self):  # faded original
        f = self._fig("Original and Modified Signal", "Wavenumber (cm-1)", "Intensity (a.u.)")
        f.add_scatter(x=self.spec.x, y=self.spec.y, mode="lines",
                      name="Original", line=dict(color="gray", width=3), opacity=0.4)
        f.add_scatter(x=self.spec.x, y=self.spec.ifft, mode="lines",
                      name="Modified", line=dict(color="royalblue", width=1.6))
        f.update_xaxes(range=[float(self.spec.x.min()), float(self.spec.x.max())])
        return f

    def _fig_diff_sig(self):
        diff = self.spec.y - self.spec.ifft
        f = self._fig("Difference Between Signals", "Wavenumber (cm-1)", "Residual Intensity")
        f.add_scatter(x=self.spec.x, y=diff, mode="lines")
        f.add_hline(y=0, line_dash="dash", line_color="black")
        f.update_xaxes(range=[float(self.spec.x.min()), float(self.spec.x.max())])
        return f

    def _draw(self):  # active views
        if self._busy or self.spec is None:
            return
        figs = []
        if self.tg_orig.value: figs.append(self._fig_original())
        if self.tg_fft.value: figs.append(self._fig_fft())
        if self.tg_lfft.value: figs.append(self._fig_lfft())
        if self.tg_mfft.value and self._applied: figs.append(self._fig_mod_fft())
        if self.tg_cmpfft.value and self._applied: figs.append(self._fig_cmp_fft())
        if self.tg_difffft.value and self._applied: figs.append(self._fig_diff_fft())
        if self.tg_msig.value and self.spec.ifft is not None: figs.append(self._fig_mod_sig())
        if self.tg_cmpsig.value and self.spec.ifft is not None: figs.append(self._fig_cmp_sig())
        if self.tg_diffsig.value and self.spec.ifft is not None: figs.append(self._fig_diff_sig())
        self._show_iframe(self.out_plot, figs, per=390)

    def show(self):  # layout
        hr = lambda: widgets.HTML("<hr style='margin:6px 0'>")
        head = lambda t: widgets.HTML(f"<h3 style='margin:4px 0'>{t}</h3>")
        toolbar = widgets.HBox([self.btn_apply, self.tg_brpoint,
                                widgets.Label("Window:"), self.dd_win, self.btn_close])
        center = widgets.HBox([self.out_norm, widgets.VBox([self.sl_delta, self.sl_rho])])
        bottom = widgets.HBox([self.sl_sigma, self.lbl_apod])
        self.panel = widgets.VBox(
            [widgets.HTML("<b style='font-family:sans-serif'>Normalized Domain</b>"),
             toolbar, center, bottom],
            layout=widgets.Layout(border="1px solid #888", padding="6px", width="1000px"))
        display(widgets.VBox([
            widgets.HTML("<h2>Spectrum Processing</h2>"),
            widgets.HBox([self.out_loader, self.btn_clear]),
            self.lbl_file, hr(),
            widgets.VBox([self.tg_orig, self.tg_fft, self.tg_lfft]), hr(),
            head("Apodization (signal processing)"),
            self.rb_mode, self.panel, hr(),
            head("Modified FFT Views"),
            widgets.VBox([self.tg_mfft, self.tg_cmpfft, self.tg_difffft]), hr(),
            head("Processed Signal Views"),
            widgets.VBox([self.tg_msig, self.tg_cmpsig, self.tg_diffsig]), hr(),
            self.btn_export, hr(),
            self.out_status, self.out_plot,
        ]))


gui = SpectrumGUI()
gui.show()